# Data PipeLine Work Book 

In [18]:
import config # config.py with db path, team colors, etc.

import os
import sys
from pathlib import Path
import pandas as pd
import numpy as np


In [19]:
# ======= BASE PATHS =======
try:
    # Works when running as a script
    base_dir = Path(__file__).resolve().parent
except NameError:
    # Fallback for notebooks or interactive mode
    base_dir = Path.cwd()

# one directory up (your config.py lives here)
config_folder = base_dir.parent

# two directories up (for TEMP, data, images)
project_root = base_dir.parent.parent

# ======= DATA FOLDERS =======
temp_folder = project_root / "TEMP"
data_folder = project_root / "data"
roster_folder = data_folder / "player_info"
school_info_folder = data_folder / "school_info"

# ======= IMAGE FOLDERS =======
img_folder = project_root / "images"
logo_folder = img_folder / "logos"
background_folder = img_folder / "background"
plot_folder = project_root / "TEMP" / "IMAGE" / "scatter_plots"


# ======= LOAD DATA =======
roster_file = roster_folder / "roster_10_30_25.csv"
roster_df = pd.read_csv(roster_file)
roster_df["Current Team"] = roster_df["Current Team"].replace("RPI", "Rensselaer")

print(roster_df.columns)

school_info_file = school_info_folder / "arena_school_info.csv"
school_info_df = pd.read_csv(school_info_file)



Index(['Current Team', 'Last_Name', 'First_Name', 'No', 'Position', 'Yr', 'Ht',
       'Wt', 'DOB', 'Hometown', 'Height_Inches', 'Draft_Year', 'NHL_Team',
       'D_Round', 'Last Team', 'League', 'City', 'State_Province', 'Country'],
      dtype='object')


## Connect To DB

In [20]:
## Connect to DB
## Connect to database using the recent_clean_db path from config.py
import sqlite3



conn = sqlite3.connect(config.recent_clean_db)
cursor = conn.cursor()
print("Connected to database:", config.recent_clean_db)

# # VERIFY CONNECTION
# # Print List of Tables
# cursor.execute("SELECT name FROM sqlite_master WHERE type='table';")
# tables = cursor.fetchall()
# print("Tables in the database:")
# for table in tables:
#     print(table[0])


Connected to database: ../../data/db/Season_YTD.db


## QUERY DB

### From Offensive Scatter Plot Notebook - OUTPUT MERGED_DF
- Average Shots
- Average goals for and against
- Shootiing Percentage
- PIMS For and Against
- Adjusted Penalty Minutes
    - Compare Overall PIMs to Adjusted PIM
- Special Teams
- Faceoff Win Percentage
- Goals vs Goals Expected, both for and against

In [21]:

#### Average SHots Per Game Taken and Allowed per Team
avg_shots_query = """
WITH UniqueGames AS (
    SELECT DISTINCT * FROM linescore
)
SELECT
    a.Team,
    AVG(a.shotsT) AS Avg_Shots_Taken,
    AVG(b.shotsT) AS Avg_Shots_Allowed
FROM UniqueGames AS a
JOIN UniqueGames AS b ON a.Game_ID = b.Game_ID AND a.Team != b.Team
GROUP BY a.Team;

"""

# Execute the query to fetch data for all teams
avg_shots_df = pd.read_sql(avg_shots_query, conn)

# Calculate average and standard deviation for "Shots Taken" and "Shots Allowed"
avg_shots_taken = avg_shots_df['Avg_Shots_Taken'].mean()
std_shots_taken = avg_shots_df['Avg_Shots_Taken'].std()
avg_shots_allowed = avg_shots_df['Avg_Shots_Allowed'].mean()
std_shots_allowed = avg_shots_df['Avg_Shots_Allowed'].std()

# Show head of dataframe
# print(avg_shots_df.head())

In [22]:
### Average Goals Per Game Scored and Allowed per Team
# SQL query to calculate average goals
avg_goals_query = """
WITH UniqueGames AS (
    SELECT DISTINCT * FROM linescore
)
SELECT 
    UG1.Team, 
    SUM(UG1.goalsT) AS Total_Goals_Scored, 
    SUM(UG2.goalsT) AS Total_Goals_Allowed,
    COUNT(DISTINCT UG1.Game_ID) AS Games_Played
FROM 
    UniqueGames UG1
JOIN 
    UniqueGames UG2 ON UG1.Game_ID = UG2.Game_ID AND UG1.Team != UG2.Team
GROUP BY 
    UG1.Team;
"""

# Execute the query and store the results in a DataFrame
avg_goals_df = pd.read_sql(avg_goals_query, conn)

# Calculate averages of goals scored and allowed
avg_goals_df['Avg_Goals_Scored'] = avg_goals_df['Total_Goals_Scored'] / avg_goals_df['Games_Played']
avg_goals_df['Avg_Goals_Allowed'] = avg_goals_df['Total_Goals_Allowed'] / avg_goals_df['Games_Played']

# Calculate averages and standard deviations
avg_goals_scored = avg_goals_df['Avg_Goals_Scored'].mean()
std_goals_scored = avg_goals_df['Avg_Goals_Scored'].std()
avg_goals_allowed = avg_goals_df['Avg_Goals_Allowed'].mean()
std_goals_allowed = avg_goals_df['Avg_Goals_Allowed'].std()

# Show head of dataframe
print(avg_goals_df.head())

               Team  Total_Goals_Scored  Total_Goals_Allowed  Games_Played  \
0         Air Force                  94                   95            30   
1            Alaska                  71                   86            27   
2  Alaska Anchorage                  48                  110            27   
3     Arizona State                  95                  116            32   
4              Army                  78                   81            30   

   Avg_Goals_Scored  Avg_Goals_Allowed  
0          3.133333           3.166667  
1          2.629630           3.185185  
2          1.777778           4.074074  
3          2.968750           3.625000  
4          2.600000           2.700000  


In [23]:
### Shooting Percentage (For / Against) per Team
# Uses goalsT / shotsT aggregated across all games (NOT per-game average of shooting%)
# This is usually what you want for a season shooting percentage.

shooting_pct_query = """
WITH UniqueGames AS (
    SELECT DISTINCT * FROM linescore
)
SELECT
    UG1.Team,
    SUM(UG1.goalsT) AS Goals_For,
    SUM(UG1.shotsT) AS Shots_For,
    SUM(UG2.goalsT) AS Goals_Against,
    SUM(UG2.shotsT) AS Shots_Against,
    COUNT(DISTINCT UG1.Game_ID) AS Games_Played
FROM UniqueGames UG1
JOIN UniqueGames UG2
    ON UG1.Game_ID = UG2.Game_ID
   AND UG1.Team != UG2.Team
GROUP BY UG1.Team;
"""

shooting_pct_df = pd.read_sql(shooting_pct_query, conn)

# Avoid divide-by-zero just in case
shooting_pct_df["ShootingPct_For"] = np.where(
    shooting_pct_df["Shots_For"] > 0,
    shooting_pct_df["Goals_For"] / shooting_pct_df["Shots_For"],
    np.nan
)

shooting_pct_df["ShootingPct_Against"] = np.where(
    shooting_pct_df["Shots_Against"] > 0,
    shooting_pct_df["Goals_Against"] / shooting_pct_df["Shots_Against"],
    np.nan
)

# Optional: format as percent points (e.g., 0.094 -> 9.4)
shooting_pct_df["ShootingPct_For_Pct"] = (shooting_pct_df["ShootingPct_For"] * 100).round(2)
shooting_pct_df["ShootingPct_Against_Pct"] = (shooting_pct_df["ShootingPct_Against"] * 100).round(2)

# League-wide averages / std devs (using the per-team values)
avg_shoot_for = shooting_pct_df["ShootingPct_For"].mean()
std_shoot_for = shooting_pct_df["ShootingPct_For"].std()
avg_shoot_against = shooting_pct_df["ShootingPct_Against"].mean()
std_shoot_against = shooting_pct_df["ShootingPct_Against"].std()

print(shooting_pct_df.sort_values("ShootingPct_For", ascending=False).head(10))


                Team  Goals_For  Shots_For  Goals_Against  Shots_Against  \
32          Michigan        141       1034             71            843   
18         Dartmouth         95        743             57            594   
49        Quinnipiac        140       1111             65            736   
56         St Thomas        113        968             87            812   
40      North Dakota        113        984             64            693   
34     Michigan Tech        113        994             89            993   
61         Wisconsin        112        992             92            803   
60  Western Michigan        117       1038             74            804   
31             Miami         92        824             83            933   
17           Cornell         82        743             49            607   

    Games_Played  ShootingPct_For  ShootingPct_Against  ShootingPct_For_Pct  \
32            30         0.136364             0.084223                13.64   
18   

In [24]:
##### Penalty Minutes Taken and Allowed per Team
# SQL query to calculate the average penalty minutes "for" and "against" each team
avg_penalty_query = """
WITH UniqueGames AS (
SELECT DISTINCT Team, Game_ID, PIM FROM linescore
)
SELECT
    a.Team,
    AVG(a.PIM) AS Avg_Penalty_Minutes_For,
    AVG(b.PIM) AS Avg_Penalty_Minutes_Against
FROM UniqueGames AS a
JOIN UniqueGames AS b ON a.Game_ID = b.Game_ID AND a.Team != b.Team
GROUP BY a.Team;
"""

# Execute the query and store the results in a DataFrame
avg_penalty_df = pd.read_sql(avg_penalty_query, conn)

# Calculate average and standard deviation for "For" and "Against"
avg_for = avg_penalty_df['Avg_Penalty_Minutes_For'].mean()
std_for = avg_penalty_df['Avg_Penalty_Minutes_For'].std()
avg_against = avg_penalty_df['Avg_Penalty_Minutes_Against'].mean()
std_against = avg_penalty_df['Avg_Penalty_Minutes_Against'].std()

# Remove 10 minute misoconducts and 

# Show head of dataframe
print(avg_penalty_df.head())

               Team  Avg_Penalty_Minutes_For  Avg_Penalty_Minutes_Against
0         Air Force                 8.233333                     8.733333
1            Alaska                11.666667                     9.037037
2  Alaska Anchorage                10.148148                     9.185185
3     Arizona State                10.875000                    11.656250
4              Army                11.433333                    10.933333


In [25]:
##### Adjusted Penalty Minutes (remove 10-minute misconducts)
# Source: pennalty_summary (Game_ID, Team, Pen_Length)
# Keeps the same "Avg For / Avg Against per game" structure as your original query.

adjusted_penalty_query = """
WITH TeamGames AS (
    -- Use linescore as the canonical list of (Team, Game_ID) so 0-PIM games are included
    SELECT DISTINCT Team, Game_ID
    FROM linescore
),

CleanPenalties AS (
    -- Convert Pen_Length to an integer minutes value (handles '2' or '2:00' style)
    SELECT
        Game_ID,
        Team,
        CASE
            WHEN instr(CAST(Pen_Length AS TEXT), ':') > 0
                THEN CAST(substr(CAST(Pen_Length AS TEXT), 1, instr(CAST(Pen_Length AS TEXT), ':') - 1) AS INTEGER)
            ELSE CAST(Pen_Length AS INTEGER)
        END AS Pen_Minutes
    FROM penalty_summary
),

TeamGameAdjPIM AS (
    -- Sum penalty minutes per team per game, excluding 10-minute penalties
    SELECT
        tg.Team,
        tg.Game_ID,
        COALESCE(SUM(CASE WHEN cp.Pen_Minutes = 10 THEN 0 ELSE cp.Pen_Minutes END), 0) AS Adj_PIM
    FROM TeamGames tg
    LEFT JOIN CleanPenalties cp
        ON tg.Game_ID = cp.Game_ID
       AND tg.Team    = cp.Team
    GROUP BY tg.Team, tg.Game_ID
),

OpponentJoin AS (
    SELECT
        a.Team,
        a.Game_ID,
        a.Adj_PIM AS Adj_PIM_For,
        b.Adj_PIM AS Adj_PIM_Against
    FROM TeamGameAdjPIM a
    JOIN TeamGameAdjPIM b
        ON a.Game_ID = b.Game_ID
       AND a.Team != b.Team
)

SELECT
    Team,
    AVG(Adj_PIM_For)     AS Adj_Penalty_Minutes_For,
    AVG(Adj_PIM_Against) AS Adj_Penalty_Minutes_Against
FROM OpponentJoin
GROUP BY Team;
"""

adj_penalty_df = pd.read_sql(adjusted_penalty_query, conn)

# League averages + std devs (mirrors your existing pattern)
adj_avg_for = adj_penalty_df["Adj_Penalty_Minutes_For"].mean()
adj_std_for = adj_penalty_df["Adj_Penalty_Minutes_For"].std()

adj_avg_against = adj_penalty_df["Adj_Penalty_Minutes_Against"].mean()
adj_std_against = adj_penalty_df["Adj_Penalty_Minutes_Against"].std()

print(adj_penalty_df.head())


               Team  Adj_Penalty_Minutes_For  Adj_Penalty_Minutes_Against
0         Air Force                 7.900000                     8.066667
1            Alaska                10.185185                     7.555556
2  Alaska Anchorage                 8.666667                     8.074074
3     Arizona State                 9.000000                     9.468750
4              Army                 9.100000                     8.933333


In [26]:
##### Merge original + adjusted penalty tables and compute differentials

penalty_compare_df = (
    avg_penalty_df.merge(
        adj_penalty_df,
        on="Team",
        how="inner"
    )
)

# Differential: how much misconducts inflate each stat
penalty_compare_df["Misconduct_Impact_For"] = (
    penalty_compare_df["Avg_Penalty_Minutes_For"]
    - penalty_compare_df["Adj_Penalty_Minutes_For"]
)

penalty_compare_df["Misconduct_Impact_Against"] = (
    penalty_compare_df["Avg_Penalty_Minutes_Against"]
    - penalty_compare_df["Adj_Penalty_Minutes_Against"]
)

# Optional but useful: percent impact
penalty_compare_df["Misconduct_Impact_For_%"] = (
    penalty_compare_df["Misconduct_Impact_For"]
    / penalty_compare_df["Avg_Penalty_Minutes_For"]
) * 100

penalty_compare_df["Misconduct_Impact_Against_%"] = (
    penalty_compare_df["Misconduct_Impact_Against"]
    / penalty_compare_df["Avg_Penalty_Minutes_Against"]
) * 100



print(penalty_compare_df.head())


               Team  Avg_Penalty_Minutes_For  Avg_Penalty_Minutes_Against  \
0         Air Force                 8.233333                     8.733333   
1            Alaska                11.666667                     9.037037   
2  Alaska Anchorage                10.148148                     9.185185   
3     Arizona State                10.875000                    11.656250   
4              Army                11.433333                    10.933333   

   Adj_Penalty_Minutes_For  Adj_Penalty_Minutes_Against  \
0                 7.900000                     8.066667   
1                10.185185                     7.555556   
2                 8.666667                     8.074074   
3                 9.000000                     9.468750   
4                 9.100000                     8.933333   

   Misconduct_Impact_For  Misconduct_Impact_Against  Misconduct_Impact_For_%  \
0               0.333333                   0.666667                 4.048583   
1               1.4814

In [27]:
##### Special Teams Efficiency per Team PP and PK
# SQL query to calculate Power Play and Penalty Kill stats
special_teams_query = """
WITH TeamPowerPlayStats AS (
    SELECT
        Team,
        Game_ID,
        SUM(PPG) AS Total_PPG,
        SUM(PPO) AS Total_PPO,
        (SUM(PPG) * 1.0 / NULLIF(SUM(PPO), 0)) * 100 AS PP_Percentage
    FROM linescore
    GROUP BY Team, Game_ID
),
OpponentPowerPlayStats AS (
    SELECT
        Team AS Opponent,
        Game_ID,
        SUM(PPG) AS Opp_Total_PPG,
        SUM(PPO) AS Opp_Total_PPO,
        (SUM(PPG) * 1.0 / NULLIF(SUM(PPO), 0)) * 100 AS Opp_PP_Percentage
    FROM linescore
    GROUP BY Team, Game_ID
)
SELECT
    t.Team,
    AVG(t.PP_Percentage) AS Team_PP_Percent,
    AVG(100 - o.Opp_PP_Percentage) AS Team_PK_Percent
FROM TeamPowerPlayStats t
LEFT JOIN OpponentPowerPlayStats o
ON t.Game_ID = o.Game_ID AND t.Team != o.Opponent
GROUP BY t.Team;
"""

# Execute the query and store the results in a DataFrame
special_teams_df = pd.read_sql(special_teams_query, conn)

# Calculate averages for trend lines
avg_pp = special_teams_df['Team_PP_Percent'].mean()
avg_pk = special_teams_df['Team_PK_Percent'].mean()
std_pp = special_teams_df['Team_PP_Percent'].std()
std_pk = special_teams_df['Team_PK_Percent'].std()

# Show head of dataframe
print(special_teams_df.head())

               Team  Team_PP_Percent  Team_PK_Percent
0         Air Force        12.881773        82.500000
1            Alaska        21.141975        80.823045
2  Alaska Anchorage        18.518519        68.635531
3     Arizona State        25.121528        79.749504
4              Army        21.285714        81.107143


In [28]:
#### Face off Win Percentage per Team
# SQL query to calculate faceoff stats
faceoff_query = """
WITH TeamFaceoffStats AS (
    SELECT
        Team,
        Game_ID,
        SUM(FOW) AS Total_FOW,
        SUM(FOL) AS Total_FOL,
        (SUM(FOW) * 1.0 / (SUM(FOW) + SUM(FOL))) AS FOW_Percentage
    FROM linescore
    GROUP BY Team, Game_ID
),
OpponentFaceoffStats AS (
    SELECT
        Team AS Opponent,
        Game_ID,
        SUM(FOW) AS Opp_Total_FOW,
        SUM(FOL) AS Opp_Total_FOL,
        (SUM(FOW) * 1.0 / (SUM(FOW) + SUM(FOL))) AS Opp_FOW_Percentage
    FROM linescore
    GROUP BY Team, Game_ID
)
SELECT
    t.Team,
    AVG(t.FOW_Percentage) AS Team_FOW_Percent,
    AVG(o.Opp_FOW_Percentage) AS Opp_FOW_Percent
FROM TeamFaceoffStats t
LEFT JOIN OpponentFaceoffStats o
ON t.Game_ID = o.Game_ID AND t.Team != o.Opponent
GROUP BY t.Team;
"""

# Execute the query and store the results in a DataFrame
faceoff_df = pd.read_sql(faceoff_query, conn)

# Calculate averages for trend lines
avg_team = faceoff_df['Team_FOW_Percent'].mean()
avg_opp = faceoff_df['Opp_FOW_Percent'].mean()
std_team = faceoff_df['Team_FOW_Percent'].std()
std_opp = faceoff_df['Opp_FOW_Percent'].std()

# Show head of dataframe
print(faceoff_df.head())

               Team  Team_FOW_Percent  Opp_FOW_Percent
0         Air Force          0.514327         0.485673
1            Alaska          0.478145         0.521855
2  Alaska Anchorage          0.446809         0.553191
3     Arizona State          0.507538         0.492462
4              Army          0.483623         0.516377


In [29]:
## Get goals scored/allowed and xG for each team each game from the linescore table
linescore_df = pd.read_sql("SELECT * FROM linescore", conn)
# print(linescore_df.head())


### HELPER FUNCTIONS DATA TRANSFORMATION
def build_goals_xgoals_shots_comparison(linescore_df: pd.DataFrame) -> pd.DataFrame:
    """
    Transform a linescore_df (2 rows per Game_ID) into:

    Game_ID, Team, goals_scored, xgoals_scored,
             goals_allowed, xgoals_allowed,
             shots_taken, shots_allowed
    """

    df = linescore_df.copy()

    # Ensure numeric types for safety
    df["goalsT"] = pd.to_numeric(df["goalsT"], errors="coerce")
    df["xG"] = pd.to_numeric(df["xG"], errors="coerce")
    df["shotsT"] = pd.to_numeric(df["shotsT"], errors="coerce")

    # Rename for clarity before merging
    df = df.rename(columns={
        "goalsT": "goals_scored",
        "xG": "xgoals_scored",
        "shotsT": "shots_taken",
    })

    # Self-merge on Game_ID to pair each team with its opponent
    merged = df.merge(
        df,
        on="Game_ID",
        suffixes=("", "_opp")
    )

    # Drop self-matches (Team vs itself)
    merged = merged[merged["Team"] != merged["Team_opp"]].copy()

    # Select and rename columns into the final structure
    final = merged[[
        "Game_ID",
        "Team",
        "goals_scored",
        "xgoals_scored",
        "shots_taken",
        "goals_scored_opp",
        "xgoals_scored_opp",
        "shots_taken_opp",
    ]].rename(columns={
        "goals_scored_opp": "goals_allowed",
        "xgoals_scored_opp": "xgoals_allowed",
        "shots_taken_opp": "shots_allowed",
    })

    # Optional: sort for sanity
    final = final.sort_values(["Game_ID", "Team"], ignore_index=True)

    return final
goals_xgoals_comparison_df = build_goals_xgoals_shots_comparison(linescore_df)
# print(goals_xgoals_comparison_df.head(10))

def aggregate_team_season_stats(team_game_df: pd.DataFrame) -> pd.DataFrame:
    """
    Aggregates game-by-game stats into season totals and per-game averages
    for each team.
    """

    df = team_game_df.copy()

    # Group by Team and aggregate sums + counts
    grouped = df.groupby("Team").agg(
        games_played=("Game_ID", "nunique"),
        goals_scored=("goals_scored", "sum"),
        xgoals_scored=("xgoals_scored", "sum"),
        goals_allowed=("goals_allowed", "sum"),
        xgoals_allowed=("xgoals_allowed", "sum"),
        shots_taken=("shots_taken", "sum"),
        shots_allowed=("shots_allowed", "sum"),
    )

    # Per-game averages
    grouped["goals_scored_per_game"] = grouped["goals_scored"] / grouped["games_played"]
    grouped["xgoals_scored_per_game"] = grouped["xgoals_scored"] / grouped["games_played"]
    grouped["goals_allowed_per_game"] = grouped["goals_allowed"] / grouped["games_played"]
    grouped["xgoals_allowed_per_game"] = grouped["xgoals_allowed"] / grouped["games_played"]
    grouped["shots_taken_per_game"] = grouped["shots_taken"] / grouped["games_played"]
    grouped["shots_allowed_per_game"] = grouped["shots_allowed"] / grouped["games_played"]
    ### Shots taken and allowed per expected goals scored/allowed

    grouped['shots_per_xgoals_scored'] = grouped['shots_taken'] / grouped['xgoals_scored']
    grouped['shots_per_xgoals_allowed'] = grouped['shots_allowed'] / grouped['xgoals_allowed']

    return grouped.reset_index()


team_season_stats_df = aggregate_team_season_stats(goals_xgoals_comparison_df)
print(team_season_stats_df.head(10))

#### Select columns from team_season_stats_df to include in merged df for plotting
cols = ["Team", "xgoals_scored_per_game", "xgoals_allowed_per_game", 
                "shots_per_xgoals_scored", "shots_per_xgoals_allowed"]

#### Filter the dataframe to include only the specified columns
xgoals_df = team_season_stats_df[cols]

# xgoals_df.head()

                Team  games_played  goals_scored  xgoals_scored  \
0          Air Force            30            94           95.0   
1             Alaska            27            71           72.8   
2   Alaska Anchorage            27            48           61.6   
3      Arizona State            32            95           88.3   
4               Army            30            78           77.3   
5          Augustana            32            94           91.6   
6      Bemidji State            32            90           97.2   
7            Bentley            31            95           87.5   
8     Boston College            28            96           96.8   
9  Boston University            31            87           92.3   

   goals_allowed  xgoals_allowed  shots_taken  shots_allowed  \
0             95            85.8          975            848   
1             86            85.1          674            833   
2            110            93.1          677            990   
3     

#### Merge Them all into single merged_df 
- This is how it was done in the off scatter plot because it is built as a single function that can be called to create any of 5 or 6 plots

In [30]:
## Merge DataFrames into a single DataFrame for plotting
merged_df = avg_shots_df.merge(avg_goals_df[['Team', 'Avg_Goals_Scored', 'Avg_Goals_Allowed']], on='Team')
merged_df = merged_df.merge(avg_penalty_df, on='Team')
merged_df = merged_df.merge(special_teams_df, on='Team')
merged_df = merged_df.merge(faceoff_df[['Team', 'Team_FOW_Percent', 'Opp_FOW_Percent']], on='Team')
merged_df = merged_df.merge(xgoals_df[['Team', 'xgoals_scored_per_game', 'xgoals_allowed_per_game', 'shots_per_xgoals_scored', 'shots_per_xgoals_allowed']])
## add shooting percentage for and against
merged_df = merged_df.merge(shooting_pct_df[['Team', 'ShootingPct_For_Pct', 'ShootingPct_Against_Pct']], on='Team')

# print(merged_df.head())
print(merged_df.columns)
# print(merged_df.info())


Index(['Team', 'Avg_Shots_Taken', 'Avg_Shots_Allowed', 'Avg_Goals_Scored',
       'Avg_Goals_Allowed', 'Avg_Penalty_Minutes_For',
       'Avg_Penalty_Minutes_Against', 'Team_PP_Percent', 'Team_PK_Percent',
       'Team_FOW_Percent', 'Opp_FOW_Percent', 'xgoals_scored_per_game',
       'xgoals_allowed_per_game', 'shots_per_xgoals_scored',
       'shots_per_xgoals_allowed', 'ShootingPct_For_Pct',
       'ShootingPct_Against_Pct'],
      dtype='object')


## From PP_Major_5v3_Exploration

In [31]:
## Load Entire tables into DataFrames
pen = pd.read_sql("SELECT * FROM penalty_summary;", conn)
goals = pd.read_sql("SELECT * FROM scoring_summary;", conn)
games = pd.read_sql("SELECT * FROM game_details;", conn)
lines = pd.read_sql("SELECT * FROM linescore;", conn)

# preserve full copies
games_full = games.copy()

### Helper Funtions
- Mostly for timeline building

In [32]:
# ----------------------------
# Team name normalization (drop-in)
# ----------------------------
TEAM_ALIAS = {
    "Rensselaer": "RPI",
    "St. Thomas": "St Thomas",
    "St. Cloud State": "St Cloud State",
    "St. Lawrence": "St Lawrence",
    # add more as you bump into them
}

def canon_team(x):
    if x is None or (isinstance(x, float) and np.isnan(x)):
        return None
    s = str(x).strip()
    return TEAM_ALIAS.get(s, s)

# Apply to all team-name columns we use
for df, cols in [
    (pen,   ["Team"]),                 # <- if your penalty table uses a different column name, change it here
    (goals, ["Team"]),
    (games, ["Home_Team", "Away_Team"]),
    (lines, ["Team"]),
]:
    for c in cols:
        if c in df.columns:
            df[c] = df[c].map(canon_team)



# ----------------------------
# Helpers: time parsing (supports 1/2/3, and optionally OT if present)
# ----------------------------
def period_start_and_len(period):
    p = str(period).strip().lower()
    # Common variants: "1st", "1", "2nd", "3", "OT", "Overtime", etc.
    if p.startswith("1"): return 0, 1200
    if p.startswith("2"): return 1200, 1200
    if p.startswith("3"): return 2400, 1200
    if "ot" in p or "over" in p: return 3600, 300   # if you want OT
    return None, None

def mmss_to_sec(t):
    if pd.isna(t): return None
    s = str(t).strip()
    if ":" not in s: return None
    m, sec = s.split(":")
    try:
        return int(m) * 60 + int(sec)
    except:
        return None

def abs_time(period, time_str, include_ot=True):
    base, plen = period_start_and_len(period)
    if base is None: 
        return None
    if (base == 3600) and (not include_ot):
        return None
    sec = mmss_to_sec(time_str)
    if sec is None:
        return None
    # Treat "20:00" / "5:00" as horn, not a second before
    if plen is not None and sec == plen:
        return base + plen
    return base + sec


### Helper function to deal with double minor edge cases
def normalize_double_minor_rows(pen_g: pd.DataFrame, team_col="Team", player_col="Player"):
    """
    Merge cases where the same player takes two 2-min penalties at the same timestamp.
    Treat as one 4-min double minor (single manpower slot, cancellable in 2-min chunks).

    Returns a new DataFrame with an added boolean column: Is_DoubleMinor
    """
    pen_g = pen_g.copy()
    pen_g["Pen_Length"] = pd.to_numeric(pen_g["Pen_Length"], errors="coerce")

    if player_col not in pen_g.columns:
        # Can't do the merge without player identity
        pen_g["Is_DoubleMinor"] = False
        return pen_g

    # Identify rows that are exactly 2-min penalties
    two = pen_g[pen_g["Pen_Length"] == 2].copy()
    if two.empty:
        pen_g["Is_DoubleMinor"] = False
        return pen_g

    # Group by same team, same player, same timestamp
    key_cols = ["Game_ID", "Period", "Time", team_col, player_col]
    counts = two.groupby(key_cols).size().rename("n").reset_index()

    # We only merge the exact pattern: two 2-min penalties
    dm_keys = counts[counts["n"] == 2]
    if dm_keys.empty:
        pen_g["Is_DoubleMinor"] = False
        return pen_g

    # Build a set of keys for fast matching
    dm_keys_tuples = set(map(tuple, dm_keys[key_cols].values.tolist()))

    # Split into: rows to merge vs keep
    def is_dm_row(r):
        return (r["Pen_Length"] == 2) and (tuple(r[c] for c in key_cols) in dm_keys_tuples)

    to_merge = pen_g[pen_g.apply(is_dm_row, axis=1)].copy()
    keep = pen_g[~pen_g.apply(is_dm_row, axis=1)].copy()

    # Create ONE 4-min row per key
    # We'll take the first row as a template and override length
    merged_rows = (
        to_merge.sort_values(key_cols)
        .groupby(key_cols, as_index=False)
        .head(1)
        .copy()
    )
    merged_rows["Pen_Length"] = 4
    merged_rows["Is_DoubleMinor"] = True

    keep["Is_DoubleMinor"] = False

    out = pd.concat([keep, merged_rows], ignore_index=True)
    return out


# ----------------------------
# Penalty typing (cancellable vs not)
# ----------------------------
def penalty_seconds(pen_min):
    # Pen_Length stored as TEXT in your DB
    try:
        return int(float(pen_min)) * 60
    except:
        return None

def is_cancellable(pen_min):
    # Majors do not cancel; minors & double minors do.
    # If you later add misconducts etc., you can refine this.
    try:
        m = int(float(pen_min))
    except:
        return False
    return m in (2, 4)


In [33]:
# Processes a single gameID to build a timeline of each team's 
# strength on the ice at an particular time

# ----------------------------
# Event-driven strength simulator for ONE game
# ----------------------------
def simulate_strength_game(game_id, home, away, pen_g, goals_g, include_ot=True):
    """
    Returns:
      intervals: list of dicts with time spans and skater counts
      events: list of dicts with goal/penalty events (optional debugging)
    """
   
    ### Helper Functions To Filter out 10, 20, ect penalties so they don't affect the manpower timeline
    def affects_manpower(pen_min):
        """
        Return True if this penalty creates a skater deficit.
        """
        try:
            m = int(float(pen_min))
        except:
            return False

        # common manpower-affecting durations
        if m in (2, 4, 5):
            return True

        # everything else we treat as non-manpower by default (10 misconduct, 20 GM, etc.)
        return False

    def penalty_seconds(pen_min):
        try:
            return int(float(pen_min)) * 60
        except:
            return None

    def is_cancellable(pen_min):
        # Only minors/double-minors cancel on a PP goal
        try:
            m = int(float(pen_min))
        except:
            return False
        return m in (2, 4)


    # Build penalty-start events
    pen_events = []
    for _, r in pen_g.iterrows():
        t0 = abs_time(r["Period"], r["Time"], include_ot=include_ot)

        pen_len = r["Pen_Length"]
        if not affects_manpower(pen_len):
            # e.g., 10-minute misconduct, game misconduct, etc. -> ignore for skater counts
            continue

        dur = penalty_seconds(pen_len)
        if t0 is None or dur is None or dur <= 0:
            continue

        team = r["Team"]
        pen_events.append({
            "t": t0,
            "type": "PEN_START",
            "team": team,
            "dur": dur,
            "cancellable": is_cancellable(pen_len),
            "raw_len": pen_len,   # optional, helps debugging
            "is_double_minor": bool(r.get("Is_DoubleMinor", False)),
        })


    # Build goal events
    goal_events = []
    for _, r in goals_g.iterrows():
        tg = abs_time(r["Period"], r["Time"], include_ot=include_ot)
        if tg is None:
            continue
        goal_events.append({
            "t": tg,
            "type": "GOAL",
            "team": r["Team"],
        })

    # Sort events by time; if same second, process penalties before goals
    def sort_key(e):
        pri = 0 if e["type"] == "PEN_START" else 1
        return (e["t"], pri)
    events = sorted(pen_events + goal_events, key=sort_key)

    # State: per team penalty queues
    # running penalties count down; queued penalties wait until a slot opens
    state = {
        home: {"running": [], "queued": []},
        away: {"running": [], "queued": []},
    }

    def running_count(team):
        return len(state[team]["running"])

    def skaters(team):
        # max 2 skaters down
        return 5 - min(2, running_count(team))

    def start_penalty(team, dur, cancellable, is_double_minor=False):
        """
        dur is seconds.
        Double minor should be cancellable in 2-minute chunks (120s).
        """
        if is_double_minor or dur == 240:
            pen_obj = {"rem": dur, "cancellable": True, "chunks": [120, 120]}
        else:
            pen_obj = {"rem": dur, "cancellable": cancellable, "chunks": None}

        if len(state[team]["running"]) < 2:
            state[team]["running"].append(pen_obj)
        else:
            state[team]["queued"].append(pen_obj)


    def tick(dt):
        # advance time by dt seconds, reduce remaining on running penalties
        for team in (home, away):
            for p in state[team]["running"]:
                p["rem"] -= dt

    def pop_expired_and_promote():
        # remove expired running; promote queued into running as slots open
        for team in (home, away):
            # remove expired
            state[team]["running"] = [p for p in state[team]["running"] if p["rem"] > 0]
            # promote queued into open slots
            while len(state[team]["running"]) < 2 and state[team]["queued"]:
                state[team]["running"].append(state[team]["queued"].pop(0))

    def next_expiration_time(now):
        # time until next running penalty expires
        times = []
        for team in (home, away):
            for p in state[team]["running"]:
                times.append(p["rem"])
        if not times:
            return None
        dt_min = min(times)
        return now + max(0, dt_min)

    def cancel_one_minor(scored_on_team):
        """
        On an advantaged goal:
        - cancel ONE cancellable penalty
        - for a double minor (chunks=[120,120]), remove only one chunk (reduce by 120s)
        """
        cancellables = [p for p in state[scored_on_team]["running"] if p.get("cancellable")]
        if not cancellables:
            return False

        # pick the cancellable penalty with the least time remaining
        pmin = min(cancellables, key=lambda p: p["rem"])

        # Double minor: reduce by one chunk (120s) instead of removing completely
        if pmin.get("chunks"):
            if pmin["chunks"]:
                pmin["chunks"].pop(0)
                pmin["rem"] -= 120

            # If no time left, remove it
            if pmin["rem"] <= 0 or not pmin["chunks"]:
                state[scored_on_team]["running"].remove(pmin)
        else:
            # Normal minor: remove entirely
            state[scored_on_team]["running"].remove(pmin)

        # Promote queued penalties if slots open
        while len(state[scored_on_team]["running"]) < 2 and state[scored_on_team]["queued"]:
            state[scored_on_team]["running"].append(state[scored_on_team]["queued"].pop(0))

        return True


    # Simulation loop
    intervals = []
    debug_events = []

    now = 0
    end_of_game = 3600 + (300 if include_ot else 0)

    i = 0
    while now <= end_of_game:
        next_event_t = events[i]["t"] if i < len(events) else None
        next_exp_t = next_expiration_time(now)

        # choose next time to jump to
        candidates = [t for t in [next_event_t, next_exp_t, end_of_game] if t is not None]
        t_next = min(candidates) if candidates else end_of_game

        if t_next > now:
            # record interval with current skater counts
            intervals.append({
                "Game_ID": game_id,
                "t_start": now,
                "t_end": t_next,
                "home": home,
                "away": away,
                "home_skaters": skaters(home),
                "away_skaters": skaters(away),
            })
            tick(t_next - now)
            now = t_next
            pop_expired_and_promote()

        # process all events at this second (pen starts then goals due to sorting)
        while i < len(events) and events[i]["t"] == now:
            e = events[i]
            if e["type"] == "PEN_START":
                start_penalty(e["team"], e["dur"], e["cancellable"], is_double_minor=e.get("is_double_minor", False))
                debug_events.append({**e, "Game_ID": game_id})
                pop_expired_and_promote()

            elif e["type"] == "GOAL":
                scoring_team = e["team"]
                other_team = home if scoring_team == away else away

                # --- CAPTURE strength BEFORE any cancellation ---
                home_sk_before = skaters(home)
                away_sk_before = skaters(away)
                scoring_sk_before = skaters(scoring_team)
                other_sk_before = skaters(other_team)

                # Determine if this goal was scored while scoring team had a manpower advantage (pre-cancel)
                adv = (scoring_sk_before > other_sk_before)

                cancelled = False
                if adv:
                    cancelled = cancel_one_minor(other_team)

                # --- CAPTURE strength AFTER cancellation ---
                home_sk_after = skaters(home)
                away_sk_after = skaters(away)

                debug_events.append({
                    **e,
                    "Game_ID": game_id,
                    "scoring_team": scoring_team,
                    "other_team": other_team,
                    "home_skaters_before": home_sk_before,
                    "away_skaters_before": away_sk_before,
                    "home_skaters_after": home_sk_after,
                    "away_skaters_after": away_sk_after,
                    "advantaged_goal": adv,
                    "cancelled_minor": cancelled,
                })
                pop_expired_and_promote()

            i += 1

        if now == end_of_game:
            break

    return intervals, debug_events


### 5v3 Opps
- Finds all 5 on 3 advantages and creates table summerizing for each team

In [34]:

# ----------------------------
# Build 5v3 opportunities across all games
# ----------------------------
pen["t_abs"] = pen.apply(lambda r: abs_time(r["Period"], r["Time"], include_ot=True), axis=1)
goals["t_abs"] = goals.apply(lambda r: abs_time(r["Period"], r["Time"], include_ot=True), axis=1)

# Keep only rows with usable time
pen2 = pen[pen["t_abs"].notna()].copy()
goals2 = goals[goals["t_abs"].notna()].copy()

opps = []

for _, gr in games.iterrows():
    gid = gr["Game_ID"]
    home = gr["Home_Team"]
    away = gr["Away_Team"]

    pen_g = pen2[pen2["Game_ID"] == gid]
    # Merge weirdly-recorded double minors (2+2 same player/time)
    pen_g = normalize_double_minor_rows(pen_g, team_col="Team", player_col="Player")
    
    goals_g = goals2[goals2["Game_ID"] == gid]

    intervals, dbg = simulate_strength_game(gid, home, away, pen_g, goals_g, include_ot=True)
    int_df = pd.DataFrame(intervals)
    if int_df.empty:
        continue

    # 5v3 for home: home has 5, away has 3
    int_df["home_5v3"] = (int_df["home_skaters"] == 5) & (int_df["away_skaters"] == 3)
    int_df["away_5v3"] = (int_df["away_skaters"] == 5) & (int_df["home_skaters"] == 3)

    # Helper: pull goals in a time window
    gg = goals_g.copy()
    gg["t_abs"] = gg["t_abs"].astype(int)

    # IMPORTANT:
    # A PP goal cancels a penalty at the *same* timestamp, so the 5v3 interval often ends at t==goal_time.
    # With half-open intervals [t_start, t_end), a goal at exactly t_end would be excluded.
    # Treat goal as occurring "just before" the state transition by shifting it back 1 second.
    gg["t_eff"] = (gg["t_abs"] - 1).clip(lower=0)

    def goals_in_window(team, t0, t1):
        return gg[(gg["Team"] == team) & (gg["t_eff"] >= t0) & (gg["t_eff"] < t1)].shape[0]

    # Identify continuous spans (opportunities)
    def add_spans(mask_col, advantaged_team, sh_team):
        sub = int_df[int_df[mask_col]].copy()
        if sub.empty:
            return
        sub = sub.sort_values("t_start")

        # merge contiguous/adjacent segments
        cur_s, cur_e = None, None
        for _, r in sub.iterrows():
            s, e = int(r["t_start"]), int(r["t_end"])
            if cur_s is None:
                cur_s, cur_e = s, e
            elif s <= cur_e:  # overlap/adjacent
                cur_e = max(cur_e, e)
            else:
                # finalize current span
                gf = goals_in_window(advantaged_team, cur_s, cur_e)
                ga = goals_in_window(sh_team, cur_s, cur_e)  # SH goals allowed during 5v3
                opps.append({
                    "Game_ID": gid,
                    "Adv_Team": advantaged_team,
                    "SH_Team": sh_team,
                    "t_start": cur_s,
                    "t_end": cur_e,
                    "duration_sec": cur_e - cur_s,
                    "GF_while_5v3": gf,
                    "GA_while_5v3": ga,
                    "Scored_on_5v3": gf > 0,
                    "Allowed_SH_on_5v3": ga > 0,
                })
                cur_s, cur_e = s, e

        # finalize last span
        gf = goals_in_window(advantaged_team, cur_s, cur_e)
        ga = goals_in_window(sh_team, cur_s, cur_e)
        opps.append({
            "Game_ID": gid,
            "Adv_Team": advantaged_team,
            "SH_Team": sh_team,
            "t_start": cur_s,
            "t_end": cur_e,
            "duration_sec": cur_e - cur_s,
            "GF_while_5v3": gf,
            "GA_while_5v3": ga,
            "Scored_on_5v3": gf > 0,
            "Allowed_SH_on_5v3": ga > 0,
        })

    add_spans("home_5v3", home, away)
    add_spans("away_5v3", away, home)

opps_53_df = pd.DataFrame(opps)

# ----------------------------
# Attach final W/L/T for advantaged team in each opportunity
# ----------------------------
lines["goalsT"] = pd.to_numeric(lines["goalsT"], errors="coerce")
score = lines[["Game_ID", "Team", "goalsT"]].copy()

opps_53_df = opps_53_df.merge(score, left_on=["Game_ID", "Adv_Team"], right_on=["Game_ID", "Team"], how="left") \
                       .rename(columns={"goalsT": "Adv_FinalGoals"}).drop(columns=["Team"], errors="ignore")

# opponent final goals
opps_53_df = opps_53_df.merge(score, left_on=["Game_ID", "SH_Team"], right_on=["Game_ID", "Team"], how="left") \
                       .rename(columns={"goalsT": "Opp_FinalGoals"}).drop(columns=["Team"], errors="ignore")

def wlt(a, b):
    if pd.isna(a) or pd.isna(b): return None
    if a > b: return "W"
    if a < b: return "L"
    return "T"

opps_53_df["Game_Result_for_Adv"] = opps_53_df.apply(lambda r: wlt(r["Adv_FinalGoals"], r["Opp_FinalGoals"]), axis=1)

# ----------------------------
# Team summary table (success + edge cases + outcomes)
# ----------------------------
if opps_53_df.empty:
    team_53_summary = pd.DataFrame()
else:
    g = opps_53_df.groupby("Adv_Team")

    team_53_summary = pd.DataFrame({
        "Opps_5v3": g.size(),
        "Opps_Scored": g["Scored_on_5v3"].sum(),
        "Opps_NoGoal": (g["Scored_on_5v3"].size() - g["Scored_on_5v3"].sum()),
        "ConvRate_anyGoal": g["Scored_on_5v3"].mean(),
        "GF_while_5v3": g["GF_while_5v3"].sum(),
        "GA_SH_while_5v3": g["GA_while_5v3"].sum(),
        "Opps_Allowed_SH": g["Allowed_SH_on_5v3"].sum(),
        "Avg_5v3_Duration_sec": g["duration_sec"].mean(),
        # “String two goals off a 5v3” proxy: scored during 5v3 AND had >=2 total goals
        # *during that same 5v3 span* (rare but catches oddities / same-second artifacts)
        "Opps_GF2plus_while_5v3": (opps_53_df["GF_while_5v3"] >= 2).groupby(opps_53_df["Adv_Team"]).sum(),
    }).fillna(0)

    # Game outcomes, split by whether they scored on the 5v3
    def wlt_counts(df):
        vc = df["Game_Result_for_Adv"].value_counts()
        return pd.Series({"W": int(vc.get("W", 0)), "L": int(vc.get("L", 0)), "T": int(vc.get("T", 0))})

    overall_wlt = opps_53_df.groupby("Adv_Team").apply(wlt_counts)
    scored_wlt = opps_53_df[opps_53_df["Scored_on_5v3"]].groupby("Adv_Team").apply(wlt_counts)
    noscore_wlt = opps_53_df[~opps_53_df["Scored_on_5v3"]].groupby("Adv_Team").apply(wlt_counts)

    # Join outcome splits
    overall_wlt.columns = [f"GameW_{c}" for c in overall_wlt.columns]  # GameW_W/GameW_L/GameW_T
    scored_wlt.columns = [f"ScoredW_{c}" for c in scored_wlt.columns]
    noscore_wlt.columns = [f"NoGoalW_{c}" for c in noscore_wlt.columns]

    team_53_summary = team_53_summary.join(overall_wlt, how="left") \
                                     .join(scored_wlt, how="left") \
                                     .join(noscore_wlt, how="left") \
                                     .fillna(0).astype({c: "int64" for c in team_53_summary.columns if c.endswith(("_W","_L","_T"))})

    # Sort by sample size then conversion
    team_53_summary = team_53_summary.sort_values(["Opps_5v3", "ConvRate_anyGoal"], ascending=[False, False])

# --- outputs ---
opps_53_df, team_53_summary




C:\Users\jbanc\AppData\Local\Temp\ipykernel_9528\4276451679.py:148: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  overall_wlt = opps_53_df.groupby("Adv_Team").apply(wlt_counts)
C:\Users\jbanc\AppData\Local\Temp\ipykernel_9528\4276451679.py:149: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  scored_wlt = opps_53_df[opps_53_df["Scored_on_5v3"]].groupby("Adv_Team").apply(wlt_counts)
C:\Users\jbanc\AppData\Local

(                                       Game_ID          Adv_Team  \
 0      2025-10-03-Connecticut-Colorado College  Colorado College   
 1           2025-10-03-Minnesota Duluth-Alaska  Minnesota Duluth   
 2            2025-10-03-Merrimack-Mass. Lowell       Mass Lowell   
 3           2025-10-04-Lake Superior-Stonehill         Stonehill   
 4           2025-10-04-Lake Superior-Stonehill     Lake Superior   
 ..                                         ...               ...   
 268  2026-02-14-Arizona State-Western Michigan  Western Michigan   
 269  2026-02-14-Arizona State-Western Michigan     Arizona State   
 270                    2026-02-14-Denver-Omaha            Denver   
 271  2026-02-14-Northern Michigan-Ferris State      Ferris State   
 272            2026-02-15-Quinnipiac-Princeton         Princeton   
 
                SH_Team  t_start  t_end  duration_sec  GF_while_5v3  \
 0          Connecticut     2129   2157            28             0   
 1               Alaska     

In [35]:

# ============================================================
# CORRECT TEAM 5v3 SUMMARY TABLE (from opps_53_df)
# ============================================================

df53 = opps_53_df.copy()

# numeric + boolean safety
num_cols = ["duration_sec", "GF_while_5v3", "GA_while_5v3"]
for c in num_cols:
    df53[c] = pd.to_numeric(df53[c], errors="coerce").fillna(0)

# ensure flags are 0/1 ints
for c in ["Scored_on_5v3", "Allowed_SH_on_5v3"]:
    if df53[c].dtype == "bool":
        df53[c] = df53[c].astype(int)
    else:
        df53[c] = pd.to_numeric(df53[c], errors="coerce").fillna(0).astype(int)

# ----------------------------
# 5v3 OFFENSE (advantaged team)
# ----------------------------
t53_for = (
    df53.groupby("Adv_Team")
    .agg(
        Opps_53_For=("Game_ID", "count"),
        Goals_53_For=("GF_while_5v3", "sum"),
        Time_53_For_sec=("duration_sec", "sum"),
        Opps_Scored=("Scored_on_5v3", "sum"),
        Opps_GF2plus=("GF_while_5v3", lambda x: (x >= 2).sum()),
    )
    .reset_index()
    .rename(columns={"Adv_Team": "Team"})
)

# ----------------------------
# 3v5 DEFENSE (short-handed team)
# IMPORTANT: goals allowed here are PP goals scored by advantaged team => GF_while_5v3
# ----------------------------
t53_against = (
    df53.groupby("SH_Team")
    .agg(
        Opps_53_Against=("Game_ID", "count"),
        Goals_53_Allowed=("GF_while_5v3", "sum"),          # <-- key fix
        Time_53_Against_sec=("duration_sec", "sum"),
        Opps_Allowed=("Scored_on_5v3", "sum"),             # <-- key fix (same event, from defender perspective)
        SHG_for_while_3v5=("GA_while_5v3", "sum"),          # short-handed goals scored while down 3v5 (rare, but real)
    )
    .reset_index()
    .rename(columns={"SH_Team": "Team"})
)

team_53_summary = t53_for.merge(t53_against, on="Team", how="outer").fillna(0)

# ----------------------------
# Derived metrics (guard divide-by-zero)
# ----------------------------
team_53_summary["ConvRate_anyGoal"] = team_53_summary["Opps_Scored"] / team_53_summary["Opps_53_For"].replace(0, np.nan)
team_53_summary["GF_per_53"] = team_53_summary["Goals_53_For"] / team_53_summary["Opps_53_For"].replace(0, np.nan)
team_53_summary["GF60_53"] = team_53_summary["Goals_53_For"] / (team_53_summary["Time_53_For_sec"] / 3600).replace(0, np.nan)

team_53_summary["Survival_53"] = 1 - (team_53_summary["Opps_Allowed"] / team_53_summary["Opps_53_Against"].replace(0, np.nan))
team_53_summary["GA_per_53"] = team_53_summary["Goals_53_Allowed"] / team_53_summary["Opps_53_Against"].replace(0, np.nan)
team_53_summary["GA60_53"] = team_53_summary["Goals_53_Allowed"] / (team_53_summary["Time_53_Against_sec"] / 3600).replace(0, np.nan)

team_53_summary = team_53_summary.fillna(0)

# sort: biggest sample first, then best finishing
team_53_summary = team_53_summary.sort_values(["Opps_53_For", "ConvRate_anyGoal"], ascending=[False, False])

team_53_summary.head(15)

,Team,Opps_53_For,Goals_53_For,Time_53_For_sec,Opps_Scored,Opps_GF2plus,Opps_53_Against,Goals_53_Allowed,Time_53_Against_sec,Opps_Allowed,SHG_for_while_3v5,ConvRate_anyGoal,GF_per_53,GF60_53,Survival_53,GA_per_53,GA60_53
22,Holy Cross,10.0,1.0,500.0,1.0,0.0,8.0,1.0,354.0,1.0,0.0,0.100000,0.100000,7.200000,0.875000,0.125000,10.169492
6,Bemidji State,9.0,3.0,487.0,3.0,0.0,2.0,1.0,81.0,1.0,0.0,0.333333,0.333333,22.176591,0.500000,0.500000,44.444444
10,Bowling Green,9.0,1.0,362.0,1.0,0.0,4.0,1.0,322.0,1.0,0.0,0.111111,0.111111,9.944751,0.750000,0.250000,11.180124
36,Minnesota Duluth,8.0,4.0,284.0,4.0,0.0,2.0,1.0,53.0,1.0,0.0,0.500000,0.500000,50.704225,0.500000,0.500000,67.924528
56,St Thomas,8.0,2.0,334.0,2.0,0.0,6.0,1.0,228.0,1.0,0.0,0.250000,0.250000,21.556886,0.833333,0.166667,15.789474
2,Alaska Anchorage,8.0,1.0,394.0,1.0,0.0,5.0,3.0,344.0,3.0,0.0,0.125000,0.125000,9.137056,0.400000,0.600000,31.395349
19,Denver,8.0,1.0,333.0,1.0,0.0,5.0,0.0,245.0,0.0,0.0,0.125000,0.125000,10.810811,1.000000,0.000000,0.000000
54,St Cloud State,7.0,4.0,374.0,4.0,0.0,5.0,1.0,281.0,1.0,0.0,0.571429,0.571429,38.502674,0.800000,0.200000,12.811388
23,Lake Superior,7.0,1.0,256.0,1.0,0.0,5.0,2.0,192.0,2.0,0.0,0.142857,0.142857,14.062500,0.600000,0.400000,37.500000
39,Niagara,7.0,1.0,448.0,1.0,0.0,10.0,3.0,525.0,3.0,0.0,0.142857,0.142857,8.035714,0.700000,0.300000,20.571429


### Majory PP Opps

In [36]:


# --- choose which column is the penalized team in pen2 ---
PEN_TEAM_COL = "Penalized_Team" if "Penalized_Team" in pen2.columns else "Team"

# 5-minute majors only (ignore misconduct 10s etc.; you already fixed manpower penalties elsewhere)
maj = pen2.copy()
maj["Pen_Length_num"] = pd.to_numeric(maj["Pen_Length"], errors="coerce")
maj = maj[(maj["Pen_Length_num"] == 5) & maj["t_abs"].notna()].copy()

# Start/end for the 5-minute window
maj["start_sec"] = maj["t_abs"].astype(int)
maj["end_sec"] = maj["start_sec"] + 300
maj["major_id"] = np.arange(len(maj))

# Add home/away so we can identify PP team
maj = maj.merge(games, on="Game_ID", how="left")

# Penalized team and PP team
maj["pen_team"] = maj[PEN_TEAM_COL]
maj["pp_team"] = np.where(
    maj["pen_team"] == maj["Home_Team"], maj["Away_Team"],
    np.where(maj["pen_team"] == maj["Away_Team"], maj["Home_Team"], None)
)

maj = maj[maj["pp_team"].notna()].copy()

# Compute window goals (ANY goals by either team during the 5:00 window)
g = goals2[goals2["t_abs"].notna()].copy()
g["t_abs"] = g["t_abs"].astype(int)

# join PP-team goals
ppg = g.merge(
    maj[["major_id", "Game_ID", "pp_team", "start_sec", "end_sec"]],
    left_on=["Game_ID", "Team"],
    right_on=["Game_ID", "pp_team"],
    how="inner",
)
ppg = ppg[(ppg["t_abs"] >= ppg["start_sec"]) & (ppg["t_abs"] < ppg["end_sec"])]
pp_counts = ppg.groupby("major_id").size().rename("GF_window")

# join SH-team (penalized team) goals during same window (these are SH goals relative to the PP team)
shg = g.merge(
    maj[["major_id", "Game_ID", "pen_team", "start_sec", "end_sec"]],
    left_on=["Game_ID", "Team"],
    right_on=["Game_ID", "pen_team"],
    how="inner",
)
shg = shg[(shg["t_abs"] >= shg["start_sec"]) & (shg["t_abs"] < shg["end_sec"])]
sh_counts = shg.groupby("major_id").size().rename("GA_window")

# assemble major-opportunity table
major_opps = maj[["major_id", "Game_ID", "pp_team", "pen_team", "start_sec", "end_sec"]].copy()
major_opps = major_opps.join(pp_counts, on="major_id").join(sh_counts, on="major_id")
major_opps[["GF_window", "GA_window"]] = major_opps[["GF_window", "GA_window"]].fillna(0).astype(int)

major_opps["Scored_on_major"] = major_opps["GF_window"] > 0
major_opps["Allowed_SH_on_major"] = major_opps["GA_window"] > 0
major_opps["Goals_Diff_window"] = major_opps["GF_window"] - major_opps["GA_window"]

# major_opps.head()

#### TOTAL MAJORS AND 5v3s IN THE DATASET WITH CONVERTION RATES ETC.

baseline_major = pd.DataFrame({
    "Total_Majors": [len(major_opps)],
    "Pct_With_Goal": [major_opps["Scored_on_major"].mean()],
    "Avg_GF_per_major": [major_opps["GF_window"].mean()],
    "Pct_2Plus_GF": [(major_opps["GF_window"] >= 2).mean()],
    "Pct_Allowed_SH": [major_opps["Allowed_SH_on_major"].mean()],
    "Avg_GoalDiff_during_window": [major_opps["Goals_Diff_window"].mean()],
})

lines2 = lines.copy()
lines2["goalsT"] = pd.to_numeric(lines2["goalsT"], errors="coerce")

maj_out = major_opps.merge(lines2, left_on=["Game_ID", "pp_team"], right_on=["Game_ID", "Team"], how="left") \
                    .rename(columns={"goalsT": "pp_goals_final"}).drop(columns=["Team"], errors="ignore")

maj_out = maj_out.merge(lines2, left_on=["Game_ID", "pen_team"], right_on=["Game_ID", "Team"], how="left") \
                 .rename(columns={"goalsT": "opp_goals_final"}).drop(columns=["Team"], errors="ignore")

def wlt(pp, opp):
    if pd.isna(pp) or pd.isna(opp): return None
    if pp > opp: return "W"
    if pp < opp: return "L"
    return "T"

maj_out["Game_Result_for_PP"] = maj_out.apply(lambda r: wlt(r["pp_goals_final"], r["opp_goals_final"]), axis=1)

baseline_major_outcomes = (
    maj_out.dropna(subset=["Game_Result_for_PP"])
           .groupby("Scored_on_major")["Game_Result_for_PP"]
           .value_counts(normalize=True)
           .unstack(fill_value=0)
)

# add sample sizes
baseline_major_outcomes["N"] = maj_out.groupby("Scored_on_major").size()

# Reorder columns
baseline_major_outcomes = baseline_major_outcomes[["N", "W", "L", "T"]]

baseline_major_outcomes



Game_Result_for_PP,N,W,L,T
Scored_on_major,,,,
False,127,0.488189,0.448819,0.062992
True,74,0.648649,0.283784,0.067568


In [37]:
# ============================================================
# TEAM MAJOR PENALTY SUMMARY TABLE (from maj_out)
# ============================================================

maj = maj_out.copy()

# numeric safety for core columns
maj["GF_window"] = pd.to_numeric(maj["GF_window"], errors="coerce").fillna(0)
maj["GA_window"] = pd.to_numeric(maj["GA_window"], errors="coerce").fillna(0)

# Convert booleans to integers explicitly (IMPORTANT)
maj["Scored_on_major"] = maj["Scored_on_major"].astype(int)
maj["Allowed_SH_on_major"] = maj["Allowed_SH_on_major"].astype(int)


# duration (seconds) derived safely
maj["start_sec"] = pd.to_numeric(maj["start_sec"], errors="coerce")
maj["end_sec"] = pd.to_numeric(maj["end_sec"], errors="coerce")
maj["Major_Duration_sec"] = (maj["end_sec"] - maj["start_sec"]).fillna(0)

# numeric safety for core columns
for c in ["GF_window", "GA_window", "Scored_on_major", "Allowed_SH_on_major"]:
    maj[c] = pd.to_numeric(maj[c], errors="coerce").fillna(0)

# ----------------------------
# FOR (team on major PP)
# ----------------------------
major_for = (
    maj.groupby("pp_team")
    .agg(
        Majors_For=("major_id", "count"),
        Goals_For=("GF_window", "sum"),
        Maj_Time_For_sec=("Major_Duration_sec", "sum"),
        Maj_1plus=("Scored_on_major", "sum"),  # assumes 0/1
        Maj_2plus=("GF_window", lambda x: (x >= 2).sum()),
    )
    .reset_index()
    .rename(columns={"pp_team": "Team"})
)

# ----------------------------
# AGAINST (team killing major)
# ----------------------------
major_against = (
    maj.groupby("pen_team")
    .agg(
        Majors_Against=("major_id", "count"),
        Goals_Against=("GA_window", "sum"),
        Maj_Time_Against_sec=("Major_Duration_sec", "sum"),
        Maj_Allow1=("Allowed_SH_on_major", "sum"),  # assumes 0/1
    )
    .reset_index()
    .rename(columns={"pen_team": "Team"})
)

team_major_summary = major_for.merge(major_against, on="Team", how="outer").fillna(0)

# derived metrics (guard divide-by-zero)
team_major_summary["GF_per_major"] = team_major_summary["Goals_For"] / team_major_summary["Majors_For"].replace(0, np.nan)
team_major_summary["GA_per_major"] = team_major_summary["Goals_Against"] / team_major_summary["Majors_Against"].replace(0, np.nan)

team_major_summary["Major_GF60"] = team_major_summary["Goals_For"] / (team_major_summary["Maj_Time_For_sec"] / 3600).replace(0, np.nan)
team_major_summary["Major_GA60"] = team_major_summary["Goals_Against"] / (team_major_summary["Maj_Time_Against_sec"] / 3600).replace(0, np.nan)

team_major_summary["Major_ScoreRate"] = team_major_summary["Maj_1plus"] / team_major_summary["Majors_For"].replace(0, np.nan)
team_major_summary["Major_MultiGoalRate"] = team_major_summary["Maj_2plus"] / team_major_summary["Majors_For"].replace(0, np.nan)
team_major_summary["Major_SurvivalRate"] = 1 - (team_major_summary["Maj_Allow1"] / team_major_summary["Majors_Against"].replace(0, np.nan))

team_major_summary = team_major_summary.fillna(0)

team_major_summary.head()


,Team,Majors_For,Goals_For,Maj_Time_For_sec,Maj_1plus,Maj_2plus,Majors_Against,Goals_Against,Maj_Time_Against_sec,Maj_Allow1,GF_per_major,GA_per_major,Major_GF60,Major_GA60,Major_ScoreRate,Major_MultiGoalRate,Major_SurvivalRate
0,Air Force,4.0,1.0,1200.0,1.0,0.0,1.0,0.0,300.0,0.0,0.25,0.00,3.0,0.0,0.25,0.0,1.00
1,Alaska,2.0,0.0,600.0,0.0,0.0,1.0,0.0,300.0,0.0,0.00,0.00,0.0,0.0,0.00,0.0,1.00
2,Alaska Anchorage,2.0,2.0,600.0,2.0,0.0,4.0,1.0,1200.0,1.0,1.00,0.25,12.0,3.0,1.00,0.0,0.75
3,Arizona State,5.0,1.0,1500.0,1.0,0.0,2.0,0.0,600.0,0.0,0.20,0.00,2.4,0.0,0.20,0.0,1.00
4,Army,2.0,0.0,600.0,0.0,0.0,5.0,0.0,1500.0,0.0,0.00,0.00,0.0,0.0,0.00,0.0,1.00


## From Point_Percentage_Year